# Orthomosaic — Part 2: Publish (Color → COG → PMTiles)

**Turn the raw orthomosaic into web-ready deliverables — end to end with GeoBrix light-tier readers, writers, and functions (no external tooling).**

Three stages, each a persisted, checkpointed data-engineering step:
1. **Color Correction** — `rx.rst_percentile_stretch` clips each RGB band to its [2, 98] percentile range of valid (non-black) pixels and rescales to uint8, removing the "hot-red" cast common in GoPro aerial imagery.
2. **Cloud-Optimised GeoTIFF (COG)** — the `cog_gbx` writer applies GDAL's COG driver (DEFLATE, 512-px internal tiles, AVERAGE overview pyramid).
3. **PMTiles** — `cog_gbx` reader → `rst_georeference` (native zoom) → `gbx_rst_xyzpyramid` (WebMercator PNG tiles) → `pmtiles_gbx` writer assembles one `.pmtiles` archive on the Volume.

> **Prerequisite.** `01a_sfm_orthomosaic` (or `01b`) must have written `orthomosaic.tif` to `output_dir`.

> **Runtime.** Runs on **Serverless environment 5** (light tier — no JAR).

---

**Last Update:** September 25, 2026

## Setup

In [ ]:
%run ./config_nb

## Stage 1 — Color Correction

Per-channel percentile stretch: each RGB band is clipped to its 2nd–98th percentile across non-black pixels and rescaled to 0–255 uint8. The correction is **persisted**, so the served COG/PMTiles carry it. Checkpointed per group; set `FORCE_CORRECTED = True` in `config_nb` to recompute.

In [ ]:
import os
import time as _t_cc
from pathlib import Path as _P
import shutil as _sh
from databricks.labs.gbx import pyrx as _pyrx

_t0 = _t_cc.perf_counter()
try:
    _groups = discover_groups()
    if not _groups:
        raise RuntimeError(f"no group_* orthomosaics under {output_dir} — run 01 first")
    for grp in _groups:
        _gp = group_paths(grp)
        _mani = _pyrx.Manifest(_gp["checkpoint"])
        _src = ortho_input(grp)
        _sig = _pyrx.input_signature({"grp": grp,
                "up": (_mani.get_sig("dense", f"{grp}::0") or _mani.get_sig("mosaic", grp) or ""),
                "src_size": (os.path.getsize(_src) if os.path.exists(_src) else 0)})
        if _pyrx.checkpoint_skip(_mani, "corrected", grp, _sig, force=bool(FORCE_CORRECTED)):
            print(f"[corrected][skip] group {grp!r} — checkpointed ({_gp['corrected']})")
            continue
        # GeoBrix rx.rst_percentile_stretch — per-band 2–98% contrast stretch to uint8,
        # a persisted data-engineering step (the served COG/PMTiles carry the correction).
        _cc_dir = f"{_P(_gp['corrected']).parent}/_corrected_{grp}"
        _P(_cc_dir).mkdir(parents=True, exist_ok=True)
        _content = _P(ortho_input(grp)).read_bytes()
        (
            spark.createDataFrame([("orthomosaic_corrected", _content)], ["source", "content"])
                 .select("source", rx.rst_fromcontent(F.col("content"), F.lit("GTiff")).alias("tile"))
                 .select("source", rx.rst_percentile_stretch("tile", F.lit(2.0), F.lit(98.0)).alias("tile"))
                 .write.format("gtiff_gbx").mode("overwrite").save(_cc_dir)
        )
        _written = sorted(_P(_cc_dir).glob("*.tif"))
        if not _written:
            raise RuntimeError(f"group {grp!r}: rst_percentile_stretch produced no .tif under {_cc_dir}")
        if _written[0].resolve() != _P(_gp["corrected"]).resolve():
            _sh.move(str(_written[0]), _gp["corrected"])
        _mani.mark_done("corrected", grp, _sig, _gp["corrected"])
        print(f"  group {grp!r}: color-corrected → {_gp['corrected']}")
    print(f"Color correction done in {_t_cc.perf_counter()-_t0:.1f}s ({len(_groups)} group(s))")
except Exception as e:
    print(f"[ERROR] Color correction failed after {_t_cc.perf_counter()-_t0:.1f}s: {e}")
    raise

### Before / after comparison

In [ ]:
import matplotlib.pyplot as plt

# VizX renders the decimated, percentile-stretched raster straight into caller-provided
# axes (ax=), so before/after is one figure — no manual thumbnailing/`imshow`.
_gp0 = group_paths(discover_groups()[0])
_g = _gp0["group"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
vz.plot_file(ortho_input(_g),   ax=axes[0], title=f"Before — uncorrected (group {_g!r})")
vz.plot_file(_gp0["corrected"], ax=axes[1], title=f"After — color-corrected (group {_g!r})")
plt.tight_layout(); plt.show()

## Stage 2 — Cloud-Optimised GeoTIFF (COG)

The GeoBrix `cog_gbx` writer converts the corrected GeoTIFF via GDAL's `driver="COG"` path (DEFLATE compression, 512-px internal tiles, AVERAGE overview pyramid). Checkpointed per group; set `FORCE_COG = True` in `config_nb` to recompute.

In [ ]:
import os
import time as _t_cog
from databricks.labs.gbx import pyrx as _pyrx

_t0 = _t_cog.perf_counter()
try:
    from pathlib import Path as _P
    import shutil as _sh
    _groups = discover_groups()
    if not _groups:
        raise RuntimeError(f"no group_* under {output_dir} — run 01/02 first")
    for grp in _groups:
        _gp = group_paths(grp)
        _mani = _pyrx.Manifest(_gp["checkpoint"])
        _src = _gp["corrected"]
        _sig = _pyrx.input_signature({"grp": grp,
                "up": (_mani.get_sig("corrected", grp) or ""),
                "src_size": (os.path.getsize(_src) if os.path.exists(_src) else 0)})
        if _pyrx.checkpoint_skip(_mani, "cog", grp, _sig, force=bool(FORCE_COG)):
            print(f"[cog][skip] group {grp!r} — checkpointed ({_gp['cog']})")
            continue
        _cog_out_dir = _gp["cog_dir"]
        _P(_cog_out_dir).mkdir(parents=True, exist_ok=True)
        # Read the corrected GeoTIFF as driver-local bytes (Volume FUSE). Spark's
        # binaryFile resolves bare paths as dbfs:, so a direct byte read + rst_fromcontent
        # writes the COG via the GeoBrix cog_gbx writer.
        _content = _P(_gp["corrected"]).read_bytes()
        (
            spark.createDataFrame([("orthomosaic_cog", _content)], ["source", "content"])
                 .select("source", rx.rst_fromcontent(F.col("content"), F.lit("GTiff")).alias("tile"))
                 .write.format("cog_gbx").mode("overwrite").save(_cog_out_dir)
        )
        # gtiff_gbx names outputs by hash; normalize to the stable per-group cog name.
        _written = sorted(_P(_cog_out_dir).glob("*.tif"))
        if not _written:
            raise RuntimeError(f"group {grp!r}: gtiff_gbx produced no .tif under {_cog_out_dir}")
        _src = next((w for w in _written if w.name == _P(_gp["cog"]).name), _written[0])
        if _src.resolve() != _P(_gp["cog"]).resolve():
            _sh.move(str(_src), _gp["cog"])
        _mani.mark_done("cog", grp, _sig, _gp["cog"])
        print(f"  group {grp!r}: COG → {_gp['cog']}")
    print(f"COG(s) written in {_t_cog.perf_counter()-_t0:.1f}s ({len(_groups)} group(s))")
except Exception as e:
    print(f"[ERROR] COG step failed after {_t_cog.perf_counter()-_t0:.1f}s: {e}")
    raise

### Preview & validate

In [ ]:
vz.plot_cog(group_paths(discover_groups()[0])["cog"])

## Stage 3 — PMTiles (interactive map serving)

All-GeoBrix light tier — no `rasterio.warp` / `pymbtiles` / `go-pmtiles`:
1. **`cog_gbx`** reader loads the COG as a raster tile.
2. **`rst_georeference`** reads its pixel scale to pick the native zoom.
3. **`gbx_rst_xyzpyramid`** (a `LATERAL` table UDF) warps it onto the WebMercatorQuad grid on the fly and emits `(z, x, y, PNG bytes)` rows.
4. **`pmtiles_gbx`** writer (`shardZoom=0`) assembles one `.pmtiles` archive directly on the Volume.

Checkpointed per group; set `FORCE_PMTILES = True` in `config_nb` to recompute.

In [ ]:
import math as _math
import os
import time as _t_pm

from databricks.labs.gbx import pyrx as _pyrx

# gbx_rst_xyzpyramid is a pyrx SQL table UDF (UDTF). config_nb's ds.register installs
# the readers/writers but not the pyrx SQL functions, so register the one UDTF we need
# on THIS session (idempotent; `only=` keeps it to a single spark.udtf.register call).
rx.register(spark, only=["gbx_rst_xyzpyramid"])

# Overview depth below the COG's native zoom, and a hard zoom cap.
_OVERVIEW_LEVELS, _MAX_Z_CAP = 4, 22

_t0 = _t_pm.perf_counter()
try:
    _groups = discover_groups()
    if not _groups:
        raise RuntimeError(f"no group_* COGs under {output_dir} — run 01-03 first")
    for grp in _groups:
        _gp = group_paths(grp)
        _mani = _pyrx.Manifest(_gp["checkpoint"])
        _src = _gp["cog"]
        _sig = _pyrx.input_signature({
            "grp": grp,
            "up": (_mani.get_sig("cog", grp) or ""),
            "src_size": (os.path.getsize(_src) if os.path.exists(_src) else 0),
        })
        if _pyrx.checkpoint_skip(_mani, "pmtiles", grp, _sig, force=bool(FORCE_PMTILES)):
            print(f"[pmtiles][skip] group {grp!r} — checkpointed ({_gp['pmtiles']})")
            continue
        # 1. Load the COG through the GeoBrix light COG reader (one whole-raster tile) —
        #    no rasterio, no manual DataFrame construction.
        _cog = spark.read.format("cog_gbx").load(_gp["cog"])
        _cog.createOrReplaceTempView("_cog_tile")
        # 2. Native max-zoom from the tile's OWN georeference (GeoBrix rst_georeference,
        #    EPSG:4326 deg/px). The cos(lat) in the WebMercator resolution and in the
        #    deg→m conversion cancel, so native zoom = log2(1.40625 / scaleX_deg).
        _sx = abs(float(
            _cog.select(rx.rst_georeference("tile").alias("gt")).first()["gt"]["scaleX"]
        ))
        _max_z = max(1, min(_MAX_Z_CAP, int(round(_math.log2(1.40625 / _sx)))))
        _min_z = max(0, _max_z - _OVERVIEW_LEVELS)
        # 3. rst_xyzpyramid (LATERAL UDTF) warps EPSG:4326 → WebMercatorQuad on the fly
        #    and emits (z, x, y, PNG bytes); pmtiles_gbx (shardZoom=0) assembles ONE
        #    .pmtiles archive straight onto the Volume (FUSE-safe sequential write).
        _tiles = spark.sql(
            "SELECT t.z, t.x, t.y, t.bytes FROM _cog_tile, "
            f"LATERAL gbx_rst_xyzpyramid(tile, {_min_z}, {_max_z}, 'PNG', 256, 'bilinear') t"
        )
        (_tiles.write.format("pmtiles_gbx").mode("overwrite")
               .option("shardZoom", "0").save(_gp["pmtiles"]))
        _mani.mark_done("pmtiles", grp, _sig, _gp["pmtiles"])
        print(f"  group {grp!r}: z{_min_z}-{_max_z} → PMTiles → {_gp['pmtiles']}")
    print(f"PMTiles for {len(_groups)} group(s) in {_t_pm.perf_counter()-_t0:.1f}s")
except Exception as e:
    print(f"[ERROR] PMTiles step failed after {_t_pm.perf_counter()-_t0:.1f}s: {e}")
    raise

### In-notebook preview

In [ ]:
vz.plot_pmtiles(group_paths(discover_groups()[0])["pmtiles"])

## Series complete — steps performed

**Stage 1 — Color Correction:** each RGB band clipped to its 2nd–98th percentile across non-black pixels and rescaled to uint8; before / after rendered side by side.

**Stage 2 — COG:** the `cog_gbx` writer applied GDAL's COG driver (DEFLATE, 512-px tiles, AVERAGE overviews); `vz.plot_cog` previewed it over a keyless basemap.

**Stage 3 — PMTiles:** `cog_gbx` → `rst_georeference` (native zoom `log2(1.40625 / scaleX)`, cos-latitude terms cancel) → `gbx_rst_xyzpyramid` (EPSG:4326 → WebMercatorQuad PNG tiles) → `pmtiles_gbx` (one `.pmtiles`); `vz.plot_pmtiles` previewed it inline.

**Deliverables.** The corrected orthomosaic GeoTIFF, COG, and PMTiles are available in `output_dir` and the configured UC Volume.

**Re-run behaviour:** every stage is checkpointed per group under `_checkpoint.json`; a re-run skips any group whose output already matches its input signature. Set `FORCE_CORRECTED` / `FORCE_COG` / `FORCE_PMTILES = True` in `config_nb` to recompute a stage.